# Finite Fields and Polynomials
## Building GF(4) from scratch, and why irreducibility is the whole story

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eduenez/courses/blob/main/2026fall/mat5173/downloads/public/activities/finite-fields-and-polynomials.ipynb)
[![License: CC BY 4.0](https://img.shields.io/badge/License-CC%20BY%204.0-lightgrey.svg)](https://creativecommons.org/licenses/by/4.0/)

---

Polynomials with coefficients in $\mathbb{F}_p$ behave a lot like ordinary
integers: they factor, and some are "prime" (irreducible). Push that analogy
one step further — quotient by an irreducible polynomial instead of a
prime number — and you get something integers alone can't give you: a
**finite field with $p^n$ elements**, for any prime power. This notebook
builds one, $\mathrm{GF}(4)$, entirely from arithmetic you already have.

1. **Polynomial arithmetic mod $p$** — addition, multiplication, and division
   with remainder, coefficient-by-coefficient.
2. **Irreducibility, by brute force** — testing and counting irreducible
   polynomials, checked against the exact known count.
3. **Building $\mathrm{GF}(4)$** — quotienting by an irreducible polynomial
   gives a genuine field: every nonzero element is invertible.
4. **What goes wrong otherwise** — the same construction with a *reducible*
   modulus, and the zero-divisor that appears because of it.

Nothing here has a grade attached and there's nothing to submit. Tinker
freely — *Runtime → Run all* always resets you to a clean state.


---
# 0. Setup


In [1]:
import itertools


---
# 1. Polynomial arithmetic mod p

A polynomial over $\mathbb{F}_p$ is just a tuple of coefficients,
`(c0, c1, c2, ...)` meaning $c_0 + c_1 x + c_2 x^2 + \cdots$, all arithmetic
done mod $p$. Division works exactly like long division of integers, using
that every nonzero element of $\mathbb{F}_p$ has an inverse (Fermat's little
theorem: $a^{-1} \equiv a^{p-2} \pmod p$).


In [2]:
def trim(a):
    a = list(a)
    while len(a) > 1 and a[-1] == 0:
        a.pop()
    return tuple(a)

def padd(a, b, p):
    n = max(len(a), len(b))
    a = a + (0,) * (n - len(a)); b = b + (0,) * (n - len(b))
    return trim(tuple((x + y) % p for x, y in zip(a, b)))

def pmul(a, b, p):
    res = [0] * (len(a) + len(b) - 1)
    for i, x in enumerate(a):
        for j, y in enumerate(b):
            res[i + j] = (res[i + j] + x * y) % p
    return trim(tuple(res))

def pdeg(a):
    a = trim(a)
    return -1 if a == (0,) else len(a) - 1

def pdivmod(a, b, p):
    """a = q*b + r (mod p), with deg(r) < deg(b)."""
    a, b = trim(a), trim(b)
    inv_lead = pow(b[-1], p - 2, p)
    q = [0] * max(len(a) - len(b) + 1, 1)
    rem = list(a)
    while True:
        rem = list(trim(tuple(rem)))
        if pdeg(tuple(rem)) < pdeg(b) or pdeg(tuple(rem)) < 0:
            break
        shift = pdeg(tuple(rem)) - pdeg(b)
        coeff = (rem[-1] * inv_lead) % p
        q[shift] = (q[shift] + coeff) % p
        for k, bc in enumerate(b):
            rem[shift + k] = (rem[shift + k] - coeff * bc) % p
    return trim(tuple(q)), trim(tuple(rem))

# 1 + x^2, divided by 1 + x, over F_5
p = 5
a, b = (1, 0, 1), (1, 1)
q, r = pdivmod(a, b, p)
print(f"({a}) = ({q}) * ({b}) + ({r})   over F_{p}")
assert padd(pmul(q, b, p), r, p) == trim(a)


((1, 0, 1)) = ((4, 1)) * ((1, 1)) + ((2,))   over F_5


**Try it:** change `p` to 2 or 3 and `a`, `b` to your own coefficient
tuples, and confirm the reconstruction line still checks out. What has to
be true of `p` for `pow(b[-1], p-2, p)` (the modular inverse step) to make
sense at all?


---
# 2. Irreducibility, by brute force

A polynomial is **irreducible** over $\mathbb{F}_p$ if it can't be written
as a product of two lower-degree polynomials — the polynomial analogue of a
prime number. For small degree, we can just try every possible factor.


In [3]:
def all_monic_polys(deg, p):
    """Every monic polynomial of the given degree over F_p."""
    if deg == 0:
        yield from ((c,) for c in range(p))
        return
    for tail in itertools.product(range(p), repeat=deg):
        yield tail + (1,)

def is_irreducible(f, p):
    d = pdeg(f)
    if d <= 0:
        return False
    for k in range(1, d // 2 + 1):
        for g in all_monic_polys(k, p):
            if pdeg(g) == 0:
                continue
            _, r = pdivmod(f, g, p)
            if trim(r) == (0,):
                return False
    return True

for p in (2, 3):
    for deg in (1, 2, 3, 4):
        count = sum(1 for f in all_monic_polys(deg, p) if is_irreducible(f, p))
        print(f"F_{p}: {count} monic irreducible polynomial(s) of degree {deg}")


F_2: 2 monic irreducible polynomial(s) of degree 1
F_2: 1 monic irreducible polynomial(s) of degree 2
F_2: 2 monic irreducible polynomial(s) of degree 3
F_2: 3 monic irreducible polynomial(s) of degree 4
F_3: 3 monic irreducible polynomial(s) of degree 1
F_3: 3 monic irreducible polynomial(s) of degree 2
F_3: 8 monic irreducible polynomial(s) of degree 3
F_3: 18 monic irreducible polynomial(s) of degree 4


**Cross-check (optional but satisfying):** the exact count of monic
irreducibles of degree $n$ over $\mathbb{F}_p$ has a closed form,
$\frac1n \sum_{d \mid n} \mu(d)\, p^{n/d}$ (Möbius function $\mu$). Look
up $\mu$, implement the formula, and confirm it matches every count above —
brute force and a 150-year-old theorem agreeing is a good feeling.


---
# 3. Building GF(4)

$\mathbb{F}_2$ has only 2 elements, so degree-2 polynomials over it have
only 4 possible coefficient pairs. Quotienting by an **irreducible** one,
$x^2 + x + 1$, turns those 4 pairs into a field with 4 elements —
$\mathrm{GF}(4)$ — where every nonzero element has a multiplicative
inverse, something $\mathbb{Z}/4\mathbb{Z}$ (also size 4, but *not* a
field) can't offer.


In [4]:
def fixed(a, n):
    """Coefficient tuple of length exactly n (pad with zeros)."""
    a = tuple(a)[:n]
    return a + (0,) * (n - len(a))

def field_mul(a, b, modulus, p):
    n = pdeg(modulus)
    prod = pmul(a, b, p)
    _, r = pdivmod(prod, modulus, p)
    return fixed(r, n)

p, n = 2, 2
elements = [fixed(e, n) for e in itertools.product(range(p), repeat=n)]
irred_mod = (1, 1, 1)   # 1 + x + x^2

print("elements of GF(4):", elements)
print()
print("multiplication table (mod x^2+x+1):")
header = "     " + "  ".join(str(b) for b in elements)
print(header)
for a in elements:
    row = [field_mul(a, b, irred_mod, p) for b in elements]
    print(f"{str(a):>8} " + "  ".join(str(r) for r in row))


elements of GF(4): [(0, 0), (0, 1), (1, 0), (1, 1)]

multiplication table (mod x^2+x+1):
     (0, 0)  (0, 1)  (1, 0)  (1, 1)
  (0, 0) (0, 0)  (0, 0)  (0, 0)  (0, 0)
  (0, 1) (0, 0)  (1, 1)  (0, 1)  (1, 0)
  (1, 0) (0, 0)  (0, 1)  (1, 0)  (1, 1)
  (1, 1) (0, 0)  (1, 0)  (1, 1)  (0, 1)


In [5]:
zero, one = fixed((0,), n), fixed((1,), n)
every_invertible = all(
    any(field_mul(a, b, irred_mod, p) == one for b in elements)
    for a in elements if a != zero
)
print("Every nonzero element has a multiplicative inverse:", every_invertible)


Every nonzero element has a multiplicative inverse: True


**Before running the next section:** the construction above only used that
$x^2+x+1$ is irreducible. What do you predict happens if we quotient by a
polynomial of the same degree that *isn't* irreducible?


---
# 4. What goes wrong with a reducible modulus

$x^2 + 1$ has the same degree as $x^2+x+1$, but over $\mathbb{F}_2$ it
factors: $x^2 + 1 = (x+1)^2$ (since $2x = 0$). Quotienting by it should *not*
give a field.


In [6]:
reducible_mod = (1, 0, 1)   # 1 + x^2 = (1+x)^2 over F_2

zero_divisors = []
for a in elements:
    for b in elements:
        if a != zero and b != zero and field_mul(a, b, reducible_mod, p) == zero:
            zero_divisors.append((a, b))

print("zero divisors found:", zero_divisors)
print("(a nonzero) * (b nonzero) = 0 is impossible in any field -- so this confirms")
print("F_2[x]/(x^2+1) is not a field, exactly because x^2+1 is reducible.")


zero divisors found: [((1, 1), (1, 1))]
(a nonzero) * (b nonzero) = 0 is impossible in any field -- so this confirms
F_2[x]/(x^2+1) is not a field, exactly because x^2+1 is reducible.


That zero divisor is $(1+x)(1+x) = 0$ — squaring a nonzero element to get
zero, which can never happen in an actual field (fields have no zero
divisors at all). Irreducibility of the modulus isn't a technical
side-condition — it's the entire reason this construction gives a field
instead of just another ring.


---
# 5. Summary

- Polynomial arithmetic mod $p$ works exactly like integer arithmetic, with
  irreducible polynomials playing the role of primes.
- Counting irreducibles by brute force matches the exact Möbius-function
  formula — this isn't a coincidence needing more sophisticated tools to
  trust, it's checkable by hand for small cases.
- $\mathbb{F}_p[x] / (\text{irreducible } f)$ is a field with $p^{\deg f}$
  elements, and *only* when $f$ is irreducible — a reducible modulus of the
  same degree gives a ring with zero divisors instead.
- This is the actual construction behind every finite field
  $\mathrm{GF}(p^n)$ you'll meet later — nothing here was a simplification
  of the real thing.


---
# 6. Exercises

1. **GF(8).** Find a degree-3 irreducible polynomial over $\mathbb{F}_2$
   (Section 2's brute force already found how many there are — list them),
   and build its multiplication table to confirm all 7 nonzero elements are
   invertible.
2. **Two different irreducibles, same field.** Over $\mathbb{F}_2$, $x^2+x+1$
   is the *only* irreducible quadratic, so GF(4) is unambiguous. Over
   $\mathbb{F}_3$ there are 3 irreducible quadratics (Section 2 found the
   count) — pick two different ones, build both fields, and check that both
   have the same number of solutions to $y^2 = -1$. (They should — they're
   isomorphic, just presented with different variable names.)
3. **Additive structure.** Confirm that adding any element of GF(4) to
   itself twice gives 0 (i.e. every element is its own additive inverse).
   Relate this to the fact that GF(4)'s additive group is $\mathbb{Z}_2
   \times \mathbb{Z}_2$, not $\mathbb{Z}_4$.
4. **A cubic that's reducible.** Find a reducible cubic over $\mathbb{F}_2$
   (one that Section 2's search did *not* count as irreducible) and show
   directly which lower-degree polynomial divides it.


---
## License

This notebook is released under the [Creative Commons Attribution 4.0
International License (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/). You are free to copy, adapt,
and redistribute it, including for commercial purposes, provided you give
appropriate credit.

Suggested attribution: *"Finite Fields and Polynomials" by [Eduardo Dueñez](https://supernumero.us/about),
licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).*
